# 01 — Data Understanding and Quality Audit

**Mục tiêu:** hiểu dữ liệu gốc, thống kê cấu trúc, kiểm tra khóa/quan hệ, duplicate, review bất thường và chất lượng dữ liệu trước khi xử lý.

### Cell code 01: Chuẩn bị môi trường và kết nối Drive

- **Mục đích:** Nạp các thư viện cơ bản, kết nối Google Drive và xác định nơi lưu dữ liệu.
- **Vì sao:** Các cell sau cần dùng chung thư viện và đường dẫn dữ liệu.
- **Giúp ích:** Notebook truy cập được dữ liệu Olist trên Drive.


In [1]:

from google.colab import drive
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
print("Đã kết nối Google Drive:", DRIVE_ROOT.exists())

# Đường dẫn chuẩn dùng xuyên suốt pipeline
DATA_DIR = DRIVE_ROOT / "BigData" / "DA-BigData" / "code" / "data"
PROCESSED_DIR = DATA_DIR / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive
Đã kết nối Google Drive: True


### Cell code 02: Tải bộ dữ liệu Olist

- **Mục đích:** Tạo thư mục data, tải bộ Olist và chép các tệp CSV vào đó.
- **Vì sao:** Cần một nguồn dữ liệu thống nhất để chạy các bước kiểm tra.
- **Giúp ích:** Có đủ các bảng gốc trong một thư mục để đọc và nối.


In [2]:
%pip -q install kagglehub

import kagglehub
import shutil

DATA_DIR = DRIVE_ROOT/"BigData"/"DA-BigData" /"code" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

source_dir = Path(
    kagglehub.dataset_download("olistbr/brazilian-ecommerce")
)

csv_files = list(source_dir.rglob("*.csv"))

if len(csv_files) != 9:
    raise RuntimeError(f"Chỉ tìm thấy {len(csv_files)} tệp CSV; cần kiểm tra bản tải xuống.")

for source_file in csv_files:
    shutil.copy2(source_file, DATA_DIR / source_file.name)

print(f"Đã lưu {len(csv_files)} tệp CSV vào: {DATA_DIR}")
for file in sorted(DATA_DIR.glob("*.csv")):
    print(file.name)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Đã lưu 9 tệp CSV vào: /content/drive/MyDrive/BigData/DA-BigData/code/data
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


### Cell code 03: Tổng quan các bảng

- **Mục đích:** Đọc các bảng bằng Dask, thống kê số dòng, số cột, dung lượng và trực quan hóa.
- **Vì sao:** Kích thước mỗi bảng khác nhau, cần biết trước khi phân tích chi tiết.
- **Giúp ích:** Xác định bảng nào lớn và kiểm tra đã nạp đủ tệp.


In [3]:
%pip -q install "dask[dataframe]" hvplot panel

import dask
import dask.dataframe as dd
import hvplot.pandas
import panel as pn

pn.extension()
pn.config.theme = "default"

def black_chart_text(plot, element):
    figure = plot.state
    figure.background_fill_color = "white"
    figure.border_fill_color = "white"
    figure.title.text_color = "black"

    for axis in list(figure.xaxis) + list(figure.yaxis):
        axis.axis_label_text_color = "black"
        axis.major_label_text_color = "black"

tables = {
    path.stem: dd.read_csv(
        str(path),
        blocksize="32MB",
        sample=2_000_000,
        assume_missing=True
    )
    for path in sorted(DATA_DIR.glob("*.csv"))
}

names = list(tables)
row_counts = dask.compute(
    *[tables[name].map_partitions(len).sum() for name in names]
)

table_summary = pd.DataFrame([
    {
        "Bảng": name,
        "Số dòng": row_count,
        "Số cột": len(tables[name].columns),
        "Số phân vùng": tables[name].npartitions,
        "Dung lượng tệp (MB)": round(
            (DATA_DIR / f"{name}.csv").stat().st_size / 1024**2, 2
        )
    }
    for name, row_count in zip(names, row_counts)
])

display(
    pn.Column(
        table_summary.hvplot.table(width=1000, height=310),
        table_summary.hvplot.bar(
            x="Bảng",
            y="Số dòng",
            width=1000,
            height=400,
            rot=45,
            title="Số dòng của từng bảng"
        ).opts(hooks=[black_chart_text]),
        styles={"background": "white", "color": "black", "padding": "12px"}
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 4.4 MB/s eta 0:00:00


/tmp/ipykernel_2812/4164959781.py:8: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column(styles={'background': 'white', ...})
    [0] HoloViews(Table, height=310, width=1000)
    [1] HoloViews(Bars, height=400, sizing_mode='fixed', width=1000)

### Cell code 04: Thống kê từng cột

- **Mục đích:** Xem kiểu dữ liệu, số giá trị thiếu và số giá trị khác nhau của từng cột.
- **Vì sao:** Những thông tin này giúp phát hiện cột khó nối hoặc cần xử lý.
- **Giúp ích:** Có cơ sở chọn khóa và các biến sẽ dùng về sau.


In [4]:
tabs = []

for name, df in tables.items():
    missing_counts, unique_counts = dask.compute(
        df.isna().sum(),
        df.nunique(dropna=True)
    )

    column_summary = pd.DataFrame({
        "Cột": df.columns,
        "Kiểu dữ liệu": df.dtypes.astype(str).values,
        "Số giá trị thiếu": missing_counts.reindex(df.columns).values,
        "Tỷ lệ thiếu (%)": (
            missing_counts.reindex(df.columns) / table_summary.loc[
                table_summary["Bảng"] == name, "Số dòng"
            ].iloc[0] * 100
        ).round(2).values,
        "Số giá trị khác nhau": unique_counts.reindex(df.columns).values
    })

    tabs.append((
        name,
        pn.Column(
            column_summary.hvplot.table(width=1000, height=360),
            column_summary.hvplot.bar(
                x="Cột",
                y="Tỷ lệ thiếu (%)",
                width=1000,
                height=360,
                rot=45,
                title=f"Tỷ lệ thiếu theo cột: {name}"
            ).opts(hooks=[black_chart_text]),
            styles={"background": "white", "color": "black", "padding": "12px"}
        )
    ))

display(pn.Tabs(*tabs))

Tabs
    [0] Column(styles={'background': 'white', ...})
        [0] HoloViews(Table, height=360, width=1000)
        [1] HoloViews(Bars, height=360, sizing_mode='fixed', width=1000)
    [1] Column(styles={'background': 'white', ...})
        [0] HoloViews(Table, height=360, width=1000)
        [1] HoloViews(Bars, height=360, sizing_mode='fixed', width=1000)
    [2] Column(styles={'background': 'white', ...})
        [0] HoloViews(Table, height=360, width=1000)
        [1] HoloViews(Bars, height=360, sizing_mode='fixed', width=1000)
    [3] Column(styles={'background': 'white', ...})
        [0] HoloViews(Table, height=360, width=1000)
        [1] HoloViews(Bars, height=360, sizing_mode='fixed', width=1000)
    [4] Column(styles={'background': 'white', ...})
        [0] HoloViews(Table, height=360, width=1000)
        [1] HoloViews(Bars, height=360, sizing_mode='fixed', width=1000)
    [5] Column(styles={'background': 'white', ...})
        [0] HoloViews(Table, height=360, width=1000)
        [1] HoloViews(Bars, height=360, sizing_mode='fixed', width=1000)
    [6] Column(styles={'background': 'white', ...})
        [0] HoloViews(Table, height=360, width=1000)
        [1] HoloViews(Bars, height=360, sizing_mode='fixed', width=1000)
    [7] Column(styles={'background': 'white', ...})
        [0] HoloViews(Table, height=360, width=1000)
        [1] HoloViews(Bars, height=360, sizing_mode='fixed', width=1000)
    [8] Column(styles={'background': 'white', ...})
        [0] HoloViews(Table, height=360, width=1000)
        [1] HoloViews(Bars, height=360, sizing_mode='fixed', width=1000)

### Cell code 05: Sơ đồ quan hệ giữa các bảng

- **Mục đích:** Kiểm tra các khóa nối và vẽ sơ đồ quan hệ bằng số liệu thực tế.
- **Vì sao:** Nối sai quan hệ một–nhiều có thể làm tăng số dòng giả.
- **Giúp ích:** Hiểu cách ghép các bảng mà không đếm trùng giao dịch.


In [5]:
import math
import dask
import pandas as pd
import networkx as nx
import holoviews as hv
import hvplot.pandas
import hvplot.networkx as hvnx
import panel as pn

pn.extension("tabulator")

names = {
    "olist_customers_dataset": "Customers",
    "olist_orders_dataset": "Orders",
    "olist_order_items_dataset": "Items",
    "olist_order_payments_dataset": "Payments",
    "olist_order_reviews_dataset": "Reviews",
    "olist_products_dataset": "Products",
    "olist_sellers_dataset": "Sellers",
    "product_category_name_translation": "Categories",
    "olist_geolocation_dataset": "Geolocation"
}

relations = [
    ("olist_orders_dataset", "olist_customers_dataset", "customer_id", "customer_id"),
    ("olist_order_items_dataset", "olist_orders_dataset", "order_id", "order_id"),
    ("olist_order_payments_dataset", "olist_orders_dataset", "order_id", "order_id"),
    ("olist_order_reviews_dataset", "olist_orders_dataset", "order_id", "order_id"),
    ("olist_order_items_dataset", "olist_products_dataset", "product_id", "product_id"),
    ("olist_order_items_dataset", "olist_sellers_dataset", "seller_id", "seller_id"),
    ("olist_products_dataset", "product_category_name_translation", "product_category_name", "product_category_name"),
    ("olist_customers_dataset", "olist_geolocation_dataset", "customer_zip_code_prefix", "geolocation_zip_code_prefix"),
    ("olist_sellers_dataset", "olist_geolocation_dataset", "seller_zip_code_prefix", "geolocation_zip_code_prefix")
]

required_keys = {
    (table_name, key)
    for source, target, source_key, target_key in relations
    for table_name, key in [(source, source_key), (target, target_key)]
}

tasks = {}

for table_name, df in tables.items():
    tasks[("rows", table_name)] = df.map_partitions(len).sum()

for table_name, key in sorted(required_keys):
    series = tables[table_name][key].dropna()

    if "zip_code_prefix" in key:
        series = series.astype("float64").astype("int64")
    else:
        series = series.astype("string")

    tasks[("valid", table_name, key)] = series.count()
    tasks[("unique", table_name, key)] = series.drop_duplicates()

results = dict(zip(tasks, dask.compute(*tasks.values())))

rows = {
    table_name: int(results[("rows", table_name)])
    for table_name in tables
}

key_sets = {
    (table_name, key): set(results[("unique", table_name, key)].tolist())
    for table_name, key in required_keys
}

stats = []

for source, target, source_key, target_key in relations:
    source_values = key_sets[(source, source_key)]
    target_values = key_sets[(target, target_key)]
    matched = len(source_values & target_values)

    stats.append({
        "Bảng nguồn": names[source],
        "Bảng đích": names[target],
        "Khóa nguồn": source_key,
        "Khóa đích": target_key,
        "Dòng nguồn": rows[source],
        "Dòng đích": rows[target],
        "Khóa nguồn duy nhất": len(source_values),
        "Khóa nguồn trùng": int(results[("valid", source, source_key)]) - len(source_values),
        "Khóa đích trùng": int(results[("valid", target, target_key)]) - len(target_values),
        "Khóa khớp": matched,
        "Khóa nguồn không khớp": len(source_values) - matched,
        "Tỷ lệ khớp (%)": round(100 * matched / len(source_values), 2) if source_values else 0
    })

relation_summary = pd.DataFrame(stats)

node_summary = pd.DataFrame([
    {"Bảng": names[table_name], "Số dòng": count}
    for table_name, count in rows.items()
])

max_matched = max(relation_summary["Khóa khớp"].max(), 1)
graph = nx.Graph()

for table_name in tables:
    graph.add_node(names[table_name])

for item in stats:
    matched = item["Khóa khớp"]

    graph.add_edge(
        item["Bảng nguồn"],
        item["Bảng đích"],
        do_rong=1.5 + 5 * math.log1p(matched) / math.log1p(max_matched),
        khoa_khop=matched
    )

main_positions = {
    "Customers": (0, 0),
    "Orders": (2.5, 0),
    "Items": (5, 0),
    "Payments": (2.5, 1.7),
    "Reviews": (2.5, -1.7),
    "Products": (7.5, 1.2),
    "Sellers": (7.5, -1.2),
    "Categories": (10, 1.2)
}

geo_positions = {
    "Customers": (0, 1),
    "Sellers": (0, -1),
    "Geolocation": (3, 0)
}

main_graph = graph.subgraph(main_positions).copy()
geo_graph = graph.subgraph(geo_positions).copy()

main_plot = hvnx.draw(
    main_graph,
    pos=main_positions,
    with_labels=True,
    node_size=5000,
    node_color="#BFDBFE",
    edge_color="#2563EB",
    edge_width=hv.dim("do_rong"),
    font_color="#000000",
    font_size="11pt",
    width=1150,
    height=480
).opts(
    bgcolor="#ffffff",
    xaxis=None,
    yaxis=None,
    padding=0.18
)

geo_plot = hvnx.draw(
    geo_graph,
    pos=geo_positions,
    with_labels=True,
    node_size=6000,
    node_color="#DDD6FE",
    edge_color="#7C3AED",
    edge_width=hv.dim("do_rong"),
    font_color="#000000",
    font_size="11pt",
    width=750,
    height=330
).opts(
    bgcolor="#ffffff",
    xaxis=None,
    yaxis=None,
    padding=0.22
)

black_text = {
    "color": "#000000",
    "background": "#ffffff"
}

table_css = ["""
:host .tabulator,
:host .tabulator * {
    color: #000000 !important;
    background-color: #ffffff !important;
}
"""]

display(
    pn.Column(
        pn.pane.Markdown("## Quan hệ giữa các bảng Olist", styles=black_text),
        pn.pane.Markdown(
            "Đường càng dày, số khóa khác nhau xuất hiện ở cả hai bảng càng lớn.",
            styles=black_text
        ),
        main_plot,
        pn.pane.Markdown("### Quan hệ với bảng Geolocation", styles=black_text),
        geo_plot,
        pn.pane.Markdown("### Số dòng thực tế", styles=black_text),
        pn.widgets.Tabulator(
            node_summary,
            show_index=False,
            theme="bootstrap5",
            width=600,
            height=285,
            stylesheets=table_css
        ),
        pn.pane.Markdown("### Kiểm tra khóa nối", styles=black_text),
        pn.widgets.Tabulator(
            relation_summary,
            show_index=False,
            theme="bootstrap5",
            width=1500,
            height=340,
            stylesheets=table_css
        ),
        styles={"background": "#ffffff", "padding": "20px"}
    )
)

/tmp/ipykernel_2812/3961459343.py:10: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension("tabulator")


Column(styles={'background': '#ffffff', ...})
    [0] Markdown(str, styles={'color': '#000000', ...})
    [1] Markdown(str, styles={'color': '#000000', ...})
    [2] HoloViews(Overlay, height=480, sizing_mode='fixed', width=1150)
    [3] Markdown(str, styles={'color': '#000000', ...})
    [4] HoloViews(Overlay, height=330, sizing_mode='fixed', width=750)
    [5] Markdown(str, styles={'color': '#000000', ...})
    [6] Tabulator(height=285, show_index=False, sizing_mode='fixed', stylesheets=['\n:host .tabulator,\...], theme='bootstrap5', value=          Bảng  ..., width=600)
    [7] Markdown(str, styles={'color': '#000000', ...})
    [8] Tabulator(height=340, show_index=False, sizing_mode='fixed', stylesheets=['\n:host .tabulator,\...], theme='bootstrap5', value=  Bảng nguồn  ..., width=1500)

### Cell code 05A: Trực quan hóa chất lượng quan hệ giữa các bảng

- **Mục đích:** Trực quan hóa tỷ lệ khóa khớp và số khóa nguồn không khớp giữa các bảng.
- **Vì sao:** Bảng `relation_summary` có nhiều cột nên khó nhận ra ngay quan hệ nào có vấn đề.
- **Giúp ích:** Nhìn nhanh chất lượng khóa nối trước khi ghép dữ liệu.


In [23]:
relation_plot = relation_summary.copy()

relation_plot["Quan hệ"] = (
    relation_plot["Bảng nguồn"].astype(str)
    + " → "
    + relation_plot["Bảng đích"].astype(str)
)

relation_plot["Tỷ lệ khớp (%)"] = pd.to_numeric(
    relation_plot["Tỷ lệ khớp (%)"],
    errors="coerce"
).fillna(0)

relation_plot["Khóa nguồn không khớp"] = pd.to_numeric(
    relation_plot["Khóa nguồn không khớp"],
    errors="coerce"
).fillna(0)

match_rate_plot = relation_plot.hvplot.bar(
    x="Quan hệ",
    y="Tỷ lệ khớp (%)",
    width=1100,
    height=430,
    rot=45,
    ylim=(0, 105),
    title="Tỷ lệ khóa nguồn khớp với bảng đích",
    xlabel="Quan hệ giữa các bảng",
    ylabel="Tỷ lệ khớp (%)"
).opts(
    hooks=[black_chart_text]
)

unmatched_plot = relation_plot.hvplot.bar(
    x="Quan hệ",
    y="Khóa nguồn không khớp",
    width=1100,
    height=430,
    rot=45,
    title="Số khóa nguồn không tìm thấy ở bảng đích",
    xlabel="Quan hệ giữa các bảng",
    ylabel="Số khóa không khớp"
).opts(
    hooks=[black_chart_text]
)

display(match_rate_plot)
display(unmatched_plot)

:Bars   [Quan hệ]   (Tỷ lệ khớp (%))

:Bars   [Quan hệ]   (Khóa nguồn không khớp)

`Thống kê giá trị tùng lặp`

### Cell code 06: Kiểm tra trùng lặp toàn bộ bảng

- **Mục đích:** Đếm dòng trùng hoàn toàn, khóa định danh trùng và khóa lặp theo quan hệ một–nhiều.
- **Vì sao:** Một mã đơn duy nhất khác với việc nhiều sản phẩm cùng thuộc một đơn.
- **Giúp ích:** Phân biệt lỗi dữ liệu với sự lặp hợp lệ trước khi làm sạch.


In [7]:
import dask
import pandas as pd
import panel as pn

pn.extension("tabulator")

key_rules = {
    "olist_customers_dataset": {
        "id": ["customer_id"],
        "repeated": [["customer_unique_id"]]
    },
    "olist_geolocation_dataset": {
        "id": None,
        "repeated": [["geolocation_zip_code_prefix"]]
    },
    "olist_order_items_dataset": {
        "id": ["order_id", "order_item_id"],
        "repeated": [["order_id"], ["product_id"], ["seller_id"]]
    },
    "olist_order_payments_dataset": {
        "id": ["order_id", "payment_sequential"],
        "repeated": [["order_id"]]
    },
    "olist_order_reviews_dataset": {
        "id": ["review_id"],
        "repeated": [["order_id"]]
    },
    "olist_orders_dataset": {
        "id": ["order_id"],
        "repeated": []
    },
    "olist_products_dataset": {
        "id": ["product_id"],
        "repeated": [["product_category_name"]]
    },
    "olist_sellers_dataset": {
        "id": ["seller_id"],
        "repeated": [["seller_zip_code_prefix"]]
    },
    "product_category_name_translation": {
        "id": ["product_category_name"],
        "repeated": []
    }
}

duplicate_rows = []
repeated_rows = []

for table_name, rules in key_rules.items():
    df = tables[table_name]
    keys_to_check = []

    if rules["id"] is not None:
        keys_to_check.append(("Định danh", rules["id"]))

    keys_to_check.extend(
        ("Lặp dự kiến", keys)
        for keys in rules["repeated"]
    )

    calculations = [
        df.map_partitions(len).sum(),
        df.drop_duplicates().map_partitions(len).sum()
    ]

    calculations.extend(
        df.dropna(subset=keys).groupby(keys).size()
        for _, keys in keys_to_check
    )

    results = dask.compute(*calculations)
    total_rows = int(results[0])
    distinct_rows = int(results[1])

    id_values = {
        "Khóa định danh": "Không có",
        "Nhóm khóa định danh trùng": None,
        "Dòng dư theo định danh": None
    }

    for (kind, keys), counts in zip(keys_to_check, results[2:]):
        repeated_groups = int((counts > 1).sum())
        extra_rows = int(counts.sum() - len(counts))

        if kind == "Định danh":
            id_values = {
                "Khóa định danh": " + ".join(keys),
                "Nhóm khóa định danh trùng": repeated_groups,
                "Dòng dư theo định danh": extra_rows
            }
        else:
            repeated_rows.append({
                "Bảng": table_name,
                "Khóa lặp theo quan hệ": " + ".join(keys),
                "Số khóa xuất hiện nhiều lần": repeated_groups,
                "Số dòng vượt quá một dòng mỗi khóa": extra_rows
            })

    duplicate_rows.append({
        "Bảng": table_name,
        "Số dòng": total_rows,
        "Dòng trùng hoàn toàn": total_rows - distinct_rows,
        **id_values
    })

duplicate_summary = pd.DataFrame(duplicate_rows)
repeated_summary = pd.DataFrame(repeated_rows)

table_css = ["""
:host .tabulator,
:host .tabulator * {
    color: #000000 !important;
    background-color: #ffffff !important;
}
"""]

display(
    pn.Column(
        pn.pane.Markdown(
            "### Dòng trùng hoàn toàn và khóa định danh trùng",
            styles={"color": "#000000"}
        ),
        pn.widgets.Tabulator(
            duplicate_summary,
            show_index=False,
            theme="bootstrap5",
            width=1250,
            height=350,
            stylesheets=table_css
        ),
        pn.pane.Markdown(
            "### Khóa lặp theo quan hệ một–nhiều",
            styles={"color": "#000000"}
        ),
        pn.widgets.Tabulator(
            repeated_summary,
            show_index=False,
            theme="bootstrap5",
            width=1250,
            height=400,
            stylesheets=table_css
        ),
        styles={"background": "#ffffff", "padding": "20px"}
    )
)

/tmp/ipykernel_2812/882350392.py:5: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension("tabulator")


Column(styles={'background': '#ffffff', ...})
    [0] Markdown(str, styles={'color': '#000000'})
    [1] Tabulator(height=350, show_index=False, sizing_mode='fixed', stylesheets=['\n:host .tabulator,\...], theme='bootstrap5', value=              ..., width=1250)
    [2] Markdown(str, styles={'color': '#000000'})
    [3] Tabulator(height=400, show_index=False, sizing_mode='fixed', stylesheets=['\n:host .tabulator,\...], theme='bootstrap5', value=              ..., width=1250)

### Cell code 06A: Trực quan hóa duplicate và khóa lặp

- **Mục đích:** Biểu diễn số dòng trùng hoàn toàn, dòng dư theo định danh và khóa lặp hợp lệ.
- **Vì sao:** Hai bảng thống kê ở Cell 06 chứa nhiều loại trùng lặp khác nhau.
- **Giúp ích:** Phân biệt nhanh duplicate thực sự với quan hệ một–nhiều hợp lệ.


In [24]:
dup_plot = duplicate_summary.copy()

dup_plot["Dòng trùng hoàn toàn"] = pd.to_numeric(
    dup_plot["Dòng trùng hoàn toàn"],
    errors="coerce"
).fillna(0)

dup_plot["Dòng dư theo định danh"] = pd.to_numeric(
    dup_plot["Dòng dư theo định danh"],
    errors="coerce"
).fillna(0)

dup_plot["Bảng"] = dup_plot["Bảng"].astype(str)

plot_full_duplicates = dup_plot.hvplot.bar(
    x="Bảng",
    y="Dòng trùng hoàn toàn",
    width=1100,
    height=430,
    rot=45,
    title="Số dòng trùng hoàn toàn theo từng bảng",
    xlabel="Bảng",
    ylabel="Số dòng trùng"
).opts(
    hooks=[black_chart_text]
)

display(plot_full_duplicates)

plot_id_duplicates = dup_plot.hvplot.bar(
    x="Bảng",
    y="Dòng dư theo định danh",
    width=1100,
    height=430,
    rot=45,
    title="Số dòng dư theo khóa định danh",
    xlabel="Bảng",
    ylabel="Số dòng dư"
).opts(
    hooks=[black_chart_text]
)

display(plot_id_duplicates)

repeat_plot = repeated_summary.copy()

repeat_plot["Số khóa xuất hiện nhiều lần"] = pd.to_numeric(
    repeat_plot["Số khóa xuất hiện nhiều lần"],
    errors="coerce"
).fillna(0)

repeat_plot["Quan hệ"] = (
    repeat_plot["Bảng"].astype(str)
    + " | "
    + repeat_plot["Khóa lặp theo quan hệ"].astype(str)
)

plot_repeated_keys = repeat_plot.hvplot.bar(
    x="Quan hệ",
    y="Số khóa xuất hiện nhiều lần",
    width=1200,
    height=460,
    rot=45,
    title="Số khóa xuất hiện nhiều lần theo quan hệ một-nhiều",
    xlabel="Quan hệ",
    ylabel="Số khóa"
).opts(
    hooks=[black_chart_text]
)

display(plot_repeated_keys)

:Bars   [Bảng]   (Dòng trùng hoàn toàn)

:Bars   [Bảng]   (Dòng dư theo định danh)

:Bars   [Quan hệ]   (Số khóa xuất hiện nhiều lần)

##Nhận xét:
- Qua thống kê, nhận thấy 8/9 bảng không có dòng trùng hoàn toàn và các khóa định danh được kiểm tra đều duy nhất, ngoại trừ bảng reviews: có 789 review_id xuất hiện nhiều lần, tạo 814 dòng vượt quá một dòng mỗi mã. Nhóm sẽ kiểm tra các review này trước khi quyết định xử lý vì chúng không phải dòng trùng hoàn toàn.
- Bảng geolocation có 261.831 dòng trùng hoàn toàn và không có khóa định danh duy nhất. Việc order_id lặp trong order_items, payments và reviews phản ánh quan hệ một–nhiều; nhóm sẽ tổng hợp các bảng này về cấp đơn hàng trước khi ghép để tránh nhân số dòng và tính sai đặc trưng.


##Kiểm tra bảng review

### Cell code 07: Xem review_id lặp

- **Mục đích:** Phân tích các review_id xuất hiện nhiều lần và các order_id liên kết.
- **Vì sao:** Không thể mặc định mỗi dòng là một lượt đánh giá mới.
- **Giúp ích:** Khoanh vùng các liên kết review cần kiểm tra sâu.


In [9]:
import pandas as pd
import panel as pn

pn.extension("tabulator")

reviews = tables["olist_order_reviews_dataset"]

review_counts = reviews.groupby("review_id").size().compute()
repeated_ids = review_counts[review_counts > 1].index.tolist()

repeated_reviews = reviews[
    reviews["review_id"].isin(repeated_ids)
].compute()

profile = repeated_reviews.groupby("review_id").agg(
    so_dong=("order_id", "size"),
    so_order_id=("order_id", lambda x: x.nunique(dropna=False))
)

distinct_rows = (
    repeated_reviews
    .drop_duplicates()
    .groupby("review_id")
    .size()
)

profile["so_ban_ghi_khac_nhau"] = distinct_rows
profile["dong_du"] = profile["so_dong"] - 1
profile["phan_loai"] = "Cùng order_id, dữ liệu khác"

profile.loc[
    profile["so_order_id"] > 1,
    "phan_loai"
] = "Gắn với nhiều order_id"

profile.loc[
    (profile["so_order_id"] == 1)
    & (profile["so_ban_ghi_khac_nhau"] == 1),
    "phan_loai"
] = "Trùng hoàn toàn"

profile = profile.reset_index().rename(columns={
    "review_id": "review_id",
    "so_dong": "Số dòng",
    "so_order_id": "Số order_id",
    "so_ban_ghi_khac_nhau": "Số bản ghi khác nhau",
    "dong_du": "Dòng dư",
    "phan_loai": "Phân loại"
})

case_summary = (
    profile.groupby("Phân loại", as_index=False)
    .agg(
        so_review_id=("review_id", "size"),
        tong_dong_du=("Dòng dư", "sum")
    )
    .rename(columns={
        "so_review_id": "Số review_id",
        "tong_dong_du": "Tổng dòng dư"
    })
)

size_summary = (
    profile.groupby(["Số dòng", "Phân loại"])
    .size()
    .reset_index(name="Số review_id")
    .sort_values(["Số dòng", "Phân loại"])
)

sample_ids = (
    profile.sort_values(["Phân loại", "Dòng dư"], ascending=[True, False])
    .groupby("Phân loại", group_keys=False)
    .head(3)["review_id"]
    .tolist()
)

sample_reviews = (
    repeated_reviews[
        repeated_reviews["review_id"].isin(sample_ids)
    ]
    .sort_values(["review_id", "order_id"])
    .reset_index(drop=True)
)

print(
    f"Review ID bị lặp: {len(profile):,} | "
    f"Dòng thuộc các ID bị lặp: {len(repeated_reviews):,} | "
    f"Dòng dư: {profile['Dòng dư'].sum():,}"
)

table_css = ["""
:host .tabulator,
:host .tabulator * {
    color: #000000 !important;
    background-color: #ffffff !important;
}
"""]

display(
    pn.Column(
        pn.pane.Markdown(
            "### Phân loại review_id bị lặp",
            styles={"color": "#000000"}
        ),
        pn.widgets.Tabulator(
            case_summary,
            show_index=False,
            theme="bootstrap5",
            width=750,
            height=180,
            stylesheets=table_css
        ),
        pn.pane.Markdown(
            "### Mỗi review_id xuất hiện bao nhiêu dòng",
            styles={"color": "#000000"}
        ),
        pn.widgets.Tabulator(
            size_summary,
            show_index=False,
            theme="bootstrap5",
            width=750,
            height=220,
            stylesheets=table_css
        ),
        pn.pane.Markdown(
            "### Chi tiết từng review_id bị lặp",
            styles={"color": "#000000"}
        ),
        pn.widgets.Tabulator(
            profile.sort_values("Dòng dư", ascending=False),
            show_index=False,
            theme="bootstrap5",
            width=1150,
            height=350,
            pagination="local",
            page_size=15,
            stylesheets=table_css
        ),
        pn.pane.Markdown(
            "### Bản ghi gốc mẫu của từng loại",
            styles={"color": "#000000"}
        ),
        pn.widgets.Tabulator(
            sample_reviews,
            show_index=False,
            theme="bootstrap5",
            width=1500,
            height=350,
            pagination="local",
            page_size=15,
            stylesheets=table_css
        ),
        styles={"background": "#ffffff", "padding": "20px"}
    )
)

/tmp/ipykernel_2812/3744271458.py:4: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension("tabulator")


Review ID bị lặp: 789 | Dòng thuộc các ID bị lặp: 1,603 | Dòng dư: 814


Column(styles={'background': '#ffffff', ...})
    [0] Markdown(str, styles={'color': '#000000'})
    [1] Tabulator(height=180, show_index=False, sizing_mode='fixed', stylesheets=['\n:host .tabulator,\...], theme='bootstrap5', value=              ..., width=750)
    [2] Markdown(str, styles={'color': '#000000'})
    [3] Tabulator(height=220, show_index=False, sizing_mode='fixed', stylesheets=['\n:host .tabulator,\...], theme='bootstrap5', value=   Số dòng    ..., width=750)
    [4] Markdown(str, styles={'color': '#000000'})
    [5] Tabulator(height=350, page_size=15, pagination='local', show_index=False, sizing_mode='fixed', stylesheets=['\n:host .tabulator,\...], theme='bootstrap5', value=              ..., width=1150)
    [6] Markdown(str, styles={'color': '#000000'})
    [7] Tabulator(height=350, page_size=15, pagination='local', show_index=False, sizing_mode='fixed', stylesheets=['\n:host .tabulator,\...], theme='bootstrap5', value=              ..., width=1500)

### Cell code 07A: Trực quan hóa review_id bị lặp

- **Mục đích:** Trực quan hóa loại review_id lặp và số lần mỗi review_id xuất hiện.
- **Vì sao:** `case_summary` và `size_summary` dễ đọc hơn khi chuyển thành biểu đồ.
- **Giúp ích:** Nhìn ngay mức độ lặp và cấu trúc các nhóm review bất thường.


In [10]:
# Cell code 07A: Trực quan hóa review_id bị lặp
import hvplot.pandas

review_type_plot = case_summary.hvplot.bar(
    x="Phân loại",
    y="Số review_id",
    width=900,
    height=400,
    rot=25,
    title="Phân loại các review_id bị lặp",
    ylabel="Số review_id"
).opts(hooks=[black_chart_text])

size_plot = (
    size_summary.groupby("Số dòng", as_index=False)["Số review_id"]
    .sum()
    .sort_values("Số dòng")
)

review_size_plot = size_plot.hvplot.bar(
    x="Số dòng",
    y="Số review_id",
    width=850,
    height=400,
    title="Phân bố số lần xuất hiện của review_id bị lặp",
    xlabel="Số dòng / review_id",
    ylabel="Số review_id"
).opts(hooks=[black_chart_text])

display(
    pn.Column(
        review_type_plot,
        review_size_plot,
        styles={"background": "white", "color": "black", "padding": "12px"}
    )
)


Column(styles={'background': 'white', ...})
    [0] HoloViews(Bars, height=400, sizing_mode='fixed', width=900)
    [1] HoloViews(Bars, height=400, sizing_mode='fixed', width=850)

##Nhận xét:
- Qua kiểm tra, có 789 review_id bị lặp đều gắn với nhiều order_id: 764 mã xuất hiện ở 2 đơn và 25 mã xuất hiện ở 3 đơn, tạo tổng cộng 814 dòng dư theo review_id. Không có dòng nào trùng hoàn toàn; các bản ghi mẫu có cùng nội dung đánh giá nhưng khác order_id.

##Kiểm tra: các đơn đó có thuộc cùng khách hàng không,

### Cell code 08: Đối chiếu khách của review lặp

- **Mục đích:** Nối reviews, orders và customers để xem các đơn cùng review_id thuộc khách nào.
- **Vì sao:** Cùng ID gắn với nhiều khách sẽ có ý nghĩa khác với nhiều đơn của một khách.
- **Giúp ích:** Làm rõ phạm vi bất thường ở cấp khách hàng.


In [11]:
from IPython.display import display

reviews = tables["olist_order_reviews_dataset"]
orders = tables["olist_orders_dataset"]
customers = tables["olist_customers_dataset"]

review_counts = reviews.groupby("review_id").size().compute()
repeated_ids = review_counts[review_counts > 1].index.tolist()

review_orders = (
    reviews[reviews["review_id"].isin(repeated_ids)]
    .merge(
        orders[
            ["order_id", "customer_id", "order_purchase_timestamp"]
        ],
        on="order_id",
        how="left"
    )
    .merge(
        customers[
            ["customer_id", "customer_unique_id"]
        ],
        on="customer_id",
        how="left"
    )
    .compute()
)

customer_check = (
    review_orders.groupby("review_id")
    .agg(
        so_dong=("order_id", "size"),
        so_order_id=("order_id", lambda x: x.nunique(dropna=True)),
        so_khach_hang=("customer_unique_id", lambda x: x.nunique(dropna=True)),
        don_khong_khop=("customer_id", lambda x: x.isna().sum()),
        khach_khong_khop=("customer_unique_id", lambda x: x.isna().sum())
    )
    .reset_index()
)

customer_check["ket_luan"] = "Cùng một khách hàng"

customer_check.loc[
    customer_check["so_khach_hang"] > 1,
    "ket_luan"
] = "Khác khách hàng"

customer_check.loc[
    (customer_check["don_khong_khop"] > 0)
    | (customer_check["khach_khong_khop"] > 0),
    "ket_luan"
] = "Chưa xác định do không khớp khóa"

summary = (
    customer_check.groupby("ket_luan", as_index=False)
    .agg(
        so_review_id=("review_id", "size"),
        so_order_id=("so_order_id", "sum")
    )
    .rename(columns={
        "ket_luan": "Kết luận",
        "so_review_id": "Số review_id",
        "so_order_id": "Tổng số đơn trong các nhóm"
    })
)

sample_ids = (
    customer_check
    .sort_values(["ket_luan", "so_dong"], ascending=[True, False])
    .groupby("ket_luan", group_keys=False)
    .head(3)["review_id"]
    .tolist()
)

sample_orders = (
    review_orders[
        review_orders["review_id"].isin(sample_ids)
    ][[
        "review_id",
        "order_id",
        "customer_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    ]]
    .sort_values(["review_id", "order_purchase_timestamp"])
    .reset_index(drop=True)
)

print(f"Số review_id được kiểm tra: {len(customer_check):,}")

print("\nTổng hợp kết quả")
display(summary)

print("\nCác review_id gắn với nhiều khách hàng hoặc không khớp khóa")
display(
    customer_check[
        customer_check["ket_luan"] != "Cùng một khách hàng"
    ].head(30)
)

print("\nMẫu đơn hàng và khách hàng của từng nhóm")
display(sample_orders)

Số review_id được kiểm tra: 789

Tổng hợp kết quả


,Kết luận,Số review_id,Tổng số đơn trong các nhóm
0,Cùng một khách hàng,789,1603



Các review_id gắn với nhiều khách hàng hoặc không khớp khóa


,review_id,so_dong,so_order_id,so_khach_hang,don_khong_khop,khach_khong_khop,ket_luan



Mẫu đơn hàng và khách hàng của từng nhóm


,review_id,order_id,customer_id,customer_unique_id,order_purchase_timestamp,review_score,review_creation_date,review_answer_timestamp
0,08528f70f579f0c830189efc523d2182,53c71d3953507c6239ff73917ed358c9,26dcb450c4b5b390e79e6d5d0f2c6535,46450c74a0d8c5ca9395da1daac6c120,2018-07-24 20:41:01,1.0,2018-08-03 00:00:00,2018-08-06 00:09:52
1,08528f70f579f0c830189efc523d2182,03310aa823a66056268a3bab36e827fb,25dbbf0c477fd4ae0880aaffbb12e8b3,46450c74a0d8c5ca9395da1daac6c120,2018-08-07 16:33:59,1.0,2018-08-03 00:00:00,2018-08-06 00:09:52
2,08528f70f579f0c830189efc523d2182,7813842ae95e8c497fc0233232ae815a,040d94f8ba8ca26014bd6f7e8a6e0c0d,46450c74a0d8c5ca9395da1daac6c120,2018-08-17 20:06:36,1.0,2018-08-03 00:00:00,2018-08-06 00:09:52
3,0c76e7a547a531e7bf9f0b99cba071c1,98c977c116f7779360e9fecffd3860b6,e144d625b739ab69821471cc91327926,74cb1ad7e6d5674325c1f99b5ea30d82,2017-08-28 20:24:52,5.0,2017-08-31 00:00:00,2017-09-05 15:27:17
4,0c76e7a547a531e7bf9f0b99cba071c1,3525e0e57f9d276d522d570bd46cb39c,780a6f1f890da521b37306aa08a6c4ea,74cb1ad7e6d5674325c1f99b5ea30d82,2017-08-28 20:24:53,5.0,2017-08-31 00:00:00,2017-09-05 15:27:17
5,0c76e7a547a531e7bf9f0b99cba071c1,16cc0fe71527d13426bdbe29205b2053,806972b159c0b485d212c2c35012c012,74cb1ad7e6d5674325c1f99b5ea30d82,2017-08-28 20:24:53,5.0,2017-08-31 00:00:00,2017-09-05 15:27:17
6,1fb4ddc969e6bea80e38deec00393a6f,d6dde74bdeb424af6b660214881b4845,137da6b7c6f269c569e4377b37b84242,6a80a86f478a93a5f4c7ba8db6441a6b,2017-07-26 17:22:33,5.0,2017-08-11 00:00:00,2017-08-12 14:35:35
7,1fb4ddc969e6bea80e38deec00393a6f,afed4265a8b956d840bc032e54dfccd1,e08f49e30ad8f0c2a6db4d6fc316c841,6a80a86f478a93a5f4c7ba8db6441a6b,2017-07-26 17:22:34,5.0,2017-08-11 00:00:00,2017-08-12 14:35:35
8,1fb4ddc969e6bea80e38deec00393a6f,3c1098cb17277b62cfc709c7a9b500f5,1e041fde4b0aa2047b6b3d7f6cbf82fb,6a80a86f478a93a5f4c7ba8db6441a6b,2017-07-26 17:22:35,5.0,2017-08-11 00:00:00,2017-08-12 14:35:35


### Cell code 08A: Trực quan hóa quan hệ review_id và khách hàng

- **Mục đích:** Trực quan hóa review_id lặp thuộc cùng hay khác khách hàng và số đơn liên kết.
- **Vì sao:** Cell 08 chủ yếu xuất bảng đối chiếu theo `customer_unique_id`.
- **Giúp ích:** Kiểm tra nhanh xem review_id lặp có bị trộn giữa nhiều khách hàng hay không.


In [12]:
# Cell code 08A: Trực quan hóa quan hệ review_id và khách hàng
import hvplot.pandas

customer_result = (
    customer_check.groupby("ket_luan")
    .size()
    .reset_index(name="Số review_id")
    .rename(columns={"ket_luan": "Kết luận"})
)

customer_result_plot = customer_result.hvplot.bar(
    x="Kết luận",
    y="Số review_id",
    width=900,
    height=400,
    rot=20,
    title="Review_id lặp thuộc cùng hay khác khách hàng",
    ylabel="Số review_id"
).opts(hooks=[black_chart_text])

order_distribution = (
    customer_check.groupby("so_order_id")
    .size()
    .reset_index(name="Số review_id")
    .rename(columns={"so_order_id": "Số order_id liên kết"})
    .sort_values("Số order_id liên kết")
)

order_distribution_plot = order_distribution.hvplot.bar(
    x="Số order_id liên kết",
    y="Số review_id",
    width=850,
    height=400,
    title="Số đơn liên kết với mỗi review_id bị lặp",
    ylabel="Số review_id"
).opts(hooks=[black_chart_text])

display(
    pn.Column(
        customer_result_plot,
        order_distribution_plot,
        styles={"background": "white", "color": "black", "padding": "12px"}
    )
)


Column(styles={'background': 'white', ...})
    [0] HoloViews(Bars, height=400, sizing_mode='fixed', width=900)
    [1] HoloViews(Bars, height=400, sizing_mode='fixed', width=850)

##Nhận xét:
- Qua kiểm tra, có 789 review_id bị lặp đều gắn với các đơn của cùng một customer_unique_id; không có nhóm nào thuộc nhiều khách hàng hoặc bị thiếu khóa nối. Vì vậy, việc lặp mã review không làm lẫn đánh giá giữa các khách hàng.

### Cell code 09: Kiểm tra thời gian review và đặt đơn

- **Mục đích:** Dùng PySpark so ngày tạo/trả lời review với ngày mua trên các ID lặp.
- **Vì sao:** Đánh giá của một đơn không thể được tạo trước khi đơn đó được đặt.
- **Giúp ích:** Phát hiện liên kết review–đơn không hợp lý về thời gian.


In [13]:
%pip -q install pyspark

from pathlib import Path
from google.colab import drive
from pyspark.sql import SparkSession

data_dir = Path("/content/drive/MyDrive/BigData/DA-BigData/code/data")

if not data_dir.exists():
    drive.mount("/content/drive")

spark = SparkSession.builder.appName("OlistReviewTimeline").getOrCreate()

reviews = (
    spark.read.option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(str(data_dir / "olist_order_reviews_dataset.csv"))
)

orders = (
    spark.read.option("header", True)
    .csv(str(data_dir / "olist_orders_dataset.csv"))
)

reviews.createOrReplaceTempView("reviews")
orders.createOrReplaceTempView("orders")

review_timeline = spark.sql("""
    WITH repeated_ids AS (
        SELECT review_id
        FROM reviews
        WHERE review_id IS NOT NULL
        GROUP BY review_id
        HAVING COUNT(*) > 1
    ),
    linked AS (
        SELECT
            r.review_id,
            r.order_id,
            o.order_purchase_timestamp,
            r.review_creation_date,
            r.review_answer_timestamp,
            to_timestamp(o.order_purchase_timestamp) AS purchased_at,
            to_date(substr(r.review_creation_date, 1, 10)) AS created_on,
            to_timestamp(r.review_answer_timestamp) AS answered_at
        FROM reviews r
        JOIN repeated_ids d ON r.review_id = d.review_id
        LEFT JOIN orders o ON r.order_id = o.order_id
    )
    SELECT
        *,
        CASE WHEN created_on < to_date(purchased_at) THEN 1 ELSE 0 END AS created_before_purchase,
        CASE WHEN answered_at < purchased_at THEN 1 ELSE 0 END AS answered_before_purchase,
        CASE WHEN created_on = to_date(purchased_at) THEN 1 ELSE 0 END AS created_same_day_as_purchase
    FROM linked
""").cache()

review_timeline.createOrReplaceTempView("review_timeline")

print("THỐNG KÊ TOÀN BỘ NHÓM REVIEW_ID LẶP")
spark.sql("""
    SELECT
        COUNT(DISTINCT review_id) AS so_nhom_review_id_lap,
        COUNT(*) AS so_dong_lien_ket,
        SUM(created_before_purchase) AS so_dong_tao_truoc_ngay_mua,
        COUNT(DISTINCT CASE WHEN created_before_purchase = 1 THEN review_id END) AS so_nhom_tao_truoc_ngay_mua,
        SUM(answered_before_purchase) AS so_dong_tra_loi_truoc_luc_mua,
        COUNT(DISTINCT CASE WHEN answered_before_purchase = 1 THEN review_id END) AS so_nhom_tra_loi_truoc_luc_mua,
        SUM(created_same_day_as_purchase) AS so_dong_tao_cung_ngay_mua,
        SUM(CASE WHEN purchased_at IS NULL THEN 1 ELSE 0 END) AS so_dong_thieu_ngay_mua,
        SUM(CASE WHEN created_on IS NULL THEN 1 ELSE 0 END) AS so_dong_thieu_ngay_tao,
        SUM(CASE WHEN answered_at IS NULL THEN 1 ELSE 0 END) AS so_dong_thieu_luc_tra_loi
    FROM review_timeline
""").show(truncate=False)

print("CÁC DÒNG CÓ REVIEW ĐƯỢC TẠO HOẶC TRẢ LỜI TRƯỚC KHI MUA")
review_timeline.filter(
    "created_before_purchase = 1 OR answered_before_purchase = 1"
).select(
    "review_id",
    "order_id",
    "order_purchase_timestamp",
    "review_creation_date",
    "review_answer_timestamp",
    "created_before_purchase",
    "answered_before_purchase"
).orderBy(
    "review_id", "order_purchase_timestamp"
).show(review_timeline.count(), truncate=False)

THỐNG KÊ TOÀN BỘ NHÓM REVIEW_ID LẶP
+---------------------+----------------+--------------------------+--------------------------+-----------------------------+-----------------------------+-------------------------+----------------------+----------------------+-------------------------+
|so_nhom_review_id_lap|so_dong_lien_ket|so_dong_tao_truoc_ngay_mua|so_nhom_tao_truoc_ngay_mua|so_dong_tra_loi_truoc_luc_mua|so_nhom_tra_loi_truoc_luc_mua|so_dong_tao_cung_ngay_mua|so_dong_thieu_ngay_mua|so_dong_thieu_ngay_tao|so_dong_thieu_luc_tra_loi|
+---------------------+----------------+--------------------------+--------------------------+-----------------------------+-----------------------------+-------------------------+----------------------+----------------------+-------------------------+
|789                  |1603            |40                        |32                        |39                           |31                           |5                        |0                     |0 

### Cell code 09A: Trực quan hóa bất thường thời gian review

- **Mục đích:** Chuyển thống kê Spark về các mốc review–mua hàng thành biểu đồ.
- **Vì sao:** Output `.show()` của Spark khó so sánh trực quan giữa nhiều loại bất thường.
- **Giúp ích:** Nhìn nhanh số dòng và số nhóm review có mốc thời gian bất hợp lý.


In [25]:
timeline_stats = spark.sql("""
    SELECT
        SUM(created_before_purchase) AS dong_review_tao_truoc_mua,

        COUNT(
            DISTINCT CASE
                WHEN created_before_purchase = 1
                THEN review_id
            END
        ) AS nhom_review_tao_truoc_mua,

        SUM(answered_before_purchase) AS dong_review_tra_loi_truoc_mua,

        COUNT(
            DISTINCT CASE
                WHEN answered_before_purchase = 1
                THEN review_id
            END
        ) AS nhom_review_tra_loi_truoc_mua,

        SUM(created_same_day_as_purchase) AS dong_review_cung_ngay_mua

    FROM review_timeline
""").toPandas()

timeline_plot = timeline_stats.T.reset_index()

timeline_plot.columns = [
    "Loại kiểm tra",
    "Số lượng"
]

timeline_plot["Loại kiểm tra"] = (
    timeline_plot["Loại kiểm tra"]
    .astype(str)
    .replace({
        "dong_review_tao_truoc_mua":
            "Dòng review tạo trước ngày mua",

        "nhom_review_tao_truoc_mua":
            "Nhóm review tạo trước ngày mua",

        "dong_review_tra_loi_truoc_mua":
            "Dòng trả lời trước lúc mua",

        "nhom_review_tra_loi_truoc_mua":
            "Nhóm trả lời trước lúc mua",

        "dong_review_cung_ngay_mua":
            "Review tạo cùng ngày mua"
    })
)

timeline_plot["Số lượng"] = pd.to_numeric(
    timeline_plot["Số lượng"],
    errors="coerce"
).fillna(0).astype(float)

plot_timeline = timeline_plot.hvplot.bar(
    x="Loại kiểm tra",
    y="Số lượng",
    width=1100,
    height=450,
    rot=35,
    title="Các bất thường về thời gian của review_id bị lặp",
    xlabel="Loại kiểm tra",
    ylabel="Số lượng"
).opts(
    hooks=[black_chart_text]
)

display(plot_timeline)

:Bars   [Loại kiểm tra]   (Số lượng)

##Nhận xét:
- Trong 789 review_id bị lặp (1.603 dòng liên kết), có 40 dòng thuộc 32 nhóm mà ngày tạo review sớm hơn ngày mua đơn liên kết; 39 dòng thuộc 31 nhóm có thời điểm trả lời sớm hơn lúc mua. Bởi vì hai nhóm số liệu này có thể trùng nhau nên không cộng lại. Không có giá trị thời gian bị thiếu.

##Kiểm tra cả 789 nhóm review lặp: với mỗi đơn liên kết, đối chiếu ngày mua, ngày giao hàng và ngày tạo review; đồng thời kiểm tra review_score có thiếu hoặc khác nhau trong cùng review_id.

### Cell code 10: Kiểm toán toàn bộ nhóm review lặp

- **Mục đích:** So ngày mua, ngày giao, ngày review và điểm số trong từng nhóm.
- **Vì sao:** Một liên kết đúng ngày mua vẫn có thể có vấn đề về giao hàng hoặc điểm số.
- **Giúp ích:** Biết nhóm nào cần xem xét trước khi dùng review làm feature.


In [15]:
from pyspark.sql import functions as F

review_audit = spark.sql("""
    WITH repeated_ids AS (
        SELECT review_id
        FROM reviews
        WHERE review_id IS NOT NULL
        GROUP BY review_id
        HAVING COUNT(*) > 1
    ),
    linked AS (
        SELECT
            r.review_id,
            r.order_id,
            r.review_score,
            o.order_status,
            o.order_purchase_timestamp,
            o.order_delivered_customer_date,
            r.review_creation_date,
            r.review_answer_timestamp,
            to_timestamp(o.order_purchase_timestamp) AS purchased_at,
            to_timestamp(o.order_delivered_customer_date) AS delivered_at,
            to_date(substr(r.review_creation_date, 1, 10)) AS review_on
        FROM reviews r
        JOIN repeated_ids d ON r.review_id = d.review_id
        LEFT JOIN orders o ON r.order_id = o.order_id
    )
    SELECT
        *,
        CASE WHEN review_on < to_date(purchased_at) THEN 1 ELSE 0 END AS review_before_purchase,
        CASE WHEN delivered_at < purchased_at THEN 1 ELSE 0 END AS delivered_before_purchase,
        CASE WHEN review_on < to_date(delivered_at) THEN 1 ELSE 0 END AS review_before_delivery,
        CASE WHEN order_status = 'delivered' AND delivered_at IS NULL THEN 1 ELSE 0 END AS delivered_order_missing_date,
        CASE WHEN review_score IS NULL OR trim(review_score) = '' THEN 1 ELSE 0 END AS missing_score,
        CASE WHEN review_score IS NOT NULL
                  AND trim(review_score) <> ''
                  AND trim(review_score) NOT RLIKE '^[1-5]$'
             THEN 1 ELSE 0 END AS invalid_score
    FROM linked
""").cache()

review_audit.createOrReplaceTempView("review_audit")

group_audit = spark.sql("""
    SELECT
        review_id,
        COUNT(*) AS so_dong,
        COUNT(DISTINCT order_id) AS so_don,
        COUNT(DISTINCT trim(review_score)) AS so_diem_khac_nhau,
        SUM(missing_score) AS so_dong_thieu_diem,
        SUM(invalid_score) AS so_dong_diem_ngoai_1_5,
        SUM(review_before_purchase) AS so_dong_review_truoc_mua,
        SUM(delivered_before_purchase) AS so_dong_giao_truoc_mua,
        SUM(review_before_delivery) AS so_dong_review_truoc_giao,
        SUM(delivered_order_missing_date) AS so_don_da_giao_thieu_ngay_giao
    FROM review_audit
    GROUP BY review_id
""").cache()

group_audit.createOrReplaceTempView("group_audit")

print("THỐNG KÊ CHO TOÀN BỘ NHÓM REVIEW_ID LẶP")
spark.sql("""
    SELECT
        COUNT(*) AS so_nhom,
        SUM(so_dong) AS so_dong_lien_ket,
        SUM(CASE WHEN so_diem_khac_nhau > 1 THEN 1 ELSE 0 END) AS nhom_co_diem_khac_nhau,
        SUM(CASE WHEN so_dong_thieu_diem > 0 THEN 1 ELSE 0 END) AS nhom_thieu_diem,
        SUM(CASE WHEN so_dong_diem_ngoai_1_5 > 0 THEN 1 ELSE 0 END) AS nhom_diem_ngoai_1_5,
        SUM(CASE WHEN so_dong_review_truoc_mua > 0 THEN 1 ELSE 0 END) AS nhom_review_truoc_mua,
        SUM(CASE WHEN so_dong_giao_truoc_mua > 0 THEN 1 ELSE 0 END) AS nhom_giao_truoc_mua,
        SUM(CASE WHEN so_dong_review_truoc_giao > 0 THEN 1 ELSE 0 END) AS nhom_review_truoc_giao,
        SUM(CASE WHEN so_don_da_giao_thieu_ngay_giao > 0 THEN 1 ELSE 0 END) AS nhom_da_giao_thieu_ngay_giao
    FROM group_audit
""").show(truncate=False)

print("KẾT QUẢ TỪNG NHÓM")
group_audit.orderBy("review_id").show(group_audit.count(), truncate=False)

print("MẪU ĐƠN LIÊN KẾT CẦN XEM XÉT")
review_audit.filter(
    (F.col("review_before_purchase") == 1) |
    (F.col("delivered_before_purchase") == 1) |
    (F.col("review_before_delivery") == 1) |
    (F.col("delivered_order_missing_date") == 1) |
    (F.col("missing_score") == 1) |
    (F.col("invalid_score") == 1)
).select(
    "review_id",
    "order_id",
    "review_score",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "review_creation_date",
    "review_before_purchase",
    "delivered_before_purchase",
    "review_before_delivery"
).orderBy("review_id", "order_purchase_timestamp").show(review_audit.count(), truncate=False)


THỐNG KÊ CHO TOÀN BỘ NHÓM REVIEW_ID LẶP
+-------+----------------+----------------------+---------------+-------------------+---------------------+-------------------+----------------------+----------------------------+
|so_nhom|so_dong_lien_ket|nhom_co_diem_khac_nhau|nhom_thieu_diem|nhom_diem_ngoai_1_5|nhom_review_truoc_mua|nhom_giao_truoc_mua|nhom_review_truoc_giao|nhom_da_giao_thieu_ngay_giao|
+-------+----------------+----------------------+---------------+-------------------+---------------------+-------------------+----------------------+----------------------------+
|789    |1603            |0                     |0              |0                  |32                   |0                  |267                   |0                           |
+-------+----------------+----------------------+---------------+-------------------+---------------------+-------------------+----------------------+----------------------------+

KẾT QUẢ TỪNG NHÓM
+--------------------------------+-------

### Cell code 10A: Trực quan hóa kết quả kiểm toán review

- **Mục đích:** Tổng hợp các loại vấn đề chất lượng của 789 nhóm review_id lặp thành biểu đồ.
- **Vì sao:** Cell 10 chứa nhiều chỉ tiêu kiểm toán ở cả cấp nhóm và cấp đơn.
- **Giúp ích:** Nhận diện nhanh loại vấn đề xuất hiện nhiều nhất trước khi quyết định cách xử lý.


In [26]:
audit_summary_plot = spark.sql("""
    SELECT
        SUM(
            CASE WHEN so_diem_khac_nhau > 1
            THEN 1 ELSE 0 END
        ) AS diem_khac_nhau,

        SUM(
            CASE WHEN so_dong_thieu_diem > 0
            THEN 1 ELSE 0 END
        ) AS thieu_diem,

        SUM(
            CASE WHEN so_dong_diem_ngoai_1_5 > 0
            THEN 1 ELSE 0 END
        ) AS diem_ngoai_1_5,

        SUM(
            CASE WHEN so_dong_review_truoc_mua > 0
            THEN 1 ELSE 0 END
        ) AS review_truoc_mua,

        SUM(
            CASE WHEN so_dong_giao_truoc_mua > 0
            THEN 1 ELSE 0 END
        ) AS giao_truoc_mua,

        SUM(
            CASE WHEN so_dong_review_truoc_giao > 0
            THEN 1 ELSE 0 END
        ) AS review_truoc_giao,

        SUM(
            CASE WHEN so_don_da_giao_thieu_ngay_giao > 0
            THEN 1 ELSE 0 END
        ) AS da_giao_thieu_ngay_giao

    FROM group_audit
""").toPandas()

audit_plot = audit_summary_plot.T.reset_index()

audit_plot.columns = [
    "Loại vấn đề",
    "Số nhóm review_id"
]

audit_plot["Loại vấn đề"] = (
    audit_plot["Loại vấn đề"]
    .astype(str)
    .replace({
        "diem_khac_nhau": "Điểm khác nhau",
        "thieu_diem": "Thiếu điểm",
        "diem_ngoai_1_5": "Điểm ngoài 1–5",
        "review_truoc_mua": "Review trước mua",
        "giao_truoc_mua": "Giao trước mua",
        "review_truoc_giao": "Review trước giao",
        "da_giao_thieu_ngay_giao":
            "Đã giao nhưng thiếu ngày giao"
    })
)

audit_plot["Số nhóm review_id"] = pd.to_numeric(
    audit_plot["Số nhóm review_id"],
    errors="coerce"
).fillna(0).astype(float)

plot_audit = audit_plot.hvplot.bar(
    x="Loại vấn đề",
    y="Số nhóm review_id",
    width=1100,
    height=450,
    rot=35,
    title="Kết quả kiểm toán 789 nhóm review_id bị lặp",
    xlabel="Loại vấn đề",
    ylabel="Số nhóm review_id"
).opts(
    hooks=[black_chart_text]
)

display(plot_audit)

:Bars   [Loại vấn đề]   (Số nhóm review_id)

##Nhận xét:
- Trong 789 nhóm review_id lặp (1.603 dòng), điểm đánh giá của mỗi nhóm đều thống nhất, không thiếu và đều nằm trong khoảng 1–5. Không có trường hợp ghi nhận giao hàng trước khi mua.
- Có 32 nhóm gắn review với đơn mua sau ngày tạo review. Ngoài ra, 267 nhóm có ít nhất một đơn mà review được tạo trước ngày giao hàng; điều này chưa đủ để kết luận dữ liệu sai, vì khách có thể đánh giá khi hàng đang giao hoặc đơn giao trễ.

##Kiểm tra các đơn trong 32 nhóm: đơn nào mua trước hoặc sau ngày tạo review, trạng thái từng đơn và ngày giao hàng.

### Cell code 11: Xem chi tiết 32 nhóm review bất thường

- **Mục đích:** Hiển thị tất cả đơn của các nhóm có review tạo trước ngày mua.
- **Vì sao:** Cần nhìn cả đơn mua trước và mua sau để hiểu một review_id bị gắn thế nào.
- **Giúp ích:** Hỗ trợ phân loại liên kết rõ ràng và liên kết chưa xác định.


In [17]:
review_32 = spark.sql("""
    SELECT
        a.review_id,
        a.order_id,
        a.review_score,
        a.review_creation_date,
        a.review_answer_timestamp,
        a.order_purchase_timestamp,
        a.order_status,
        a.order_delivered_customer_date,
        CASE
            WHEN to_date(a.purchased_at) < a.review_on THEN 'Mua trước review'
            WHEN to_date(a.purchased_at) = a.review_on THEN 'Mua cùng ngày review'
            WHEN to_date(a.purchased_at) > a.review_on THEN 'Mua sau review'
            ELSE 'Thiếu ngày mua'
        END AS thu_tu_mua_review,
        a.review_before_delivery
    FROM review_audit a
    JOIN group_audit g ON a.review_id = g.review_id
    WHERE g.so_dong_review_truoc_mua > 0
""").cache()

review_32.createOrReplaceTempView("review_32")

print("TỔNG HỢP 32 NHÓM")
spark.sql("""
    SELECT
        review_id,
        COUNT(*) AS so_don_lien_ket,
        SUM(CASE WHEN thu_tu_mua_review = 'Mua trước review' THEN 1 ELSE 0 END) AS so_don_mua_truoc_review,
        SUM(CASE WHEN thu_tu_mua_review = 'Mua cùng ngày review' THEN 1 ELSE 0 END) AS so_don_mua_cung_ngay,
        SUM(CASE WHEN thu_tu_mua_review = 'Mua sau review' THEN 1 ELSE 0 END) AS so_don_mua_sau_review,
        SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS so_don_da_giao,
        SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS so_don_da_huy,
        SUM(review_before_delivery) AS so_don_review_truoc_giao
    FROM review_32
    GROUP BY review_id
    ORDER BY review_id
""").show(32, truncate=False)

print("CHI TIẾT TẤT CẢ ĐƠN TRONG 32 NHÓM")
review_32.orderBy(
    "review_id", "order_purchase_timestamp"
).show(review_32.count(), truncate=False)

TỔNG HỢP 32 NHÓM
+--------------------------------+---------------+-----------------------+--------------------+---------------------+--------------+-------------+------------------------+
|review_id                       |so_don_lien_ket|so_don_mua_truoc_review|so_don_mua_cung_ngay|so_don_mua_sau_review|so_don_da_giao|so_don_da_huy|so_don_review_truoc_giao|
+--------------------------------+---------------+-----------------------+--------------------+---------------------+--------------+-------------+------------------------+
|047fd109ced39e02296f6aeb74f6a6f1|2              |0                      |0                   |2                    |0             |2            |0                       |
|08528f70f579f0c830189efc523d2182|3              |1                      |0                   |2                    |1             |2            |0                       |
|1775bd097716d67420d128df008252eb|2              |1                      |0                   |1                    |1     

### Cell code 11A: Trực quan hóa 32 nhóm review bất thường

- **Mục đích:** Trực quan hóa thứ tự mua–review và trạng thái đơn trong 32 nhóm bất thường.
- **Vì sao:** Bảng chi tiết của Cell 11 rất dài và khó nhận ra phân bố tổng thể.
- **Giúp ích:** Thấy ngay bao nhiêu đơn mua trước/sau review và trạng thái của các đơn liên quan.


In [18]:
# Cell code 11A: Trực quan hóa 32 nhóm review bất thường
import hvplot.pandas

review_order_timing = (
    review_32.groupBy("thu_tu_mua_review")
    .count()
    .toPandas()
    .rename(columns={"thu_tu_mua_review": "Quan hệ thời gian", "count": "Số đơn"})
    .sort_values("Số đơn", ascending=False)
)

review_order_timing_plot = review_order_timing.hvplot.bar(
    x="Quan hệ thời gian",
    y="Số đơn",
    width=900,
    height=400,
    rot=20,
    title="Thứ tự mua hàng và thời điểm tạo review trong 32 nhóm",
    ylabel="Số đơn"
).opts(hooks=[black_chart_text])

status_32 = (
    review_32.groupBy("order_status")
    .count()
    .toPandas()
    .rename(columns={"order_status": "Trạng thái đơn", "count": "Số đơn"})
    .sort_values("Số đơn", ascending=False)
)

status_32_plot = status_32.hvplot.bar(
    x="Trạng thái đơn",
    y="Số đơn",
    width=900,
    height=400,
    rot=25,
    title="Trạng thái đơn hàng trong 32 nhóm review bất thường",
    ylabel="Số đơn"
).opts(hooks=[black_chart_text])

display(
    pn.Column(
        review_order_timing_plot,
        status_32_plot,
        styles={"background": "white", "color": "black", "padding": "12px"}
    )
)


Column(styles={'background': 'white', ...})
    [0] HoloViews(Bars, height=400, sizing_mode='fixed', width=900)
    [1] HoloViews(Bars, height=400, sizing_mode='fixed', width=900)

##Nhận xét:
- Có 29/32 nhóm có một đơn mua trước ngày tạo review và đơn khác mua sau đó. 2 nhóm chỉ liên kết với các đơn mua sau review; 1 nhóm có đơn mua cùng ngày nên chưa xác định được thứ tự chính xác.
- Có 40 đơn mua sau ngày tạo review. Phần lớn là đơn đã hủy, nhưng bảng chi tiết cũng có đơn đã giao, đang vận chuyển.

##Thống kê từng khách hàng theo id và liên kết qua lịch sử mua hàng của tưng khách hàng đó nhóm theo đơn hàng từ đó sau đó từ những đơn hàng đó thống kê có bao nhiêu review id

### Cell code 12: Theo dõi lịch sử mua của khách có review lặp

- **Mục đích:** Liệt kê đơn và review_id theo trình tự mua của từng khách liên quan.
- **Vì sao:** Lịch sử của khách giúp đối chiếu review với các đơn khác cùng người.
- **Giúp ích:** Thấy các trường hợp ID cũ xuất hiện lại ở đơn mua sau.


In [19]:
from pathlib import Path

customer_path = Path(data_dir) / "olist_customers_dataset.csv"

if not customer_path.exists():
    customer_path = next(
        Path("/content/drive").rglob("olist_customers_dataset.csv")
    )

customers = (
    spark.read.option("header", True)
    .csv(str(customer_path))
)

customers.createOrReplaceTempView("customers")

customer_order_history = spark.sql("""
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        o.order_status,
        o.order_delivered_customer_date,
        COUNT(r.review_id) AS so_dong_review,
        COUNT(DISTINCT r.review_id) AS so_review_id_cua_don,
        sort_array(collect_set(r.review_id)) AS danh_sach_review_id,
        MIN(r.review_creation_date) AS review_tao_som_nhat,
        MAX(r.review_creation_date) AS review_tao_muon_nhat
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    LEFT JOIN reviews r ON o.order_id = r.order_id
    GROUP BY
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        o.order_status,
        o.order_delivered_customer_date
""").cache()

customer_order_history.createOrReplaceTempView("customer_order_history")

customer_review_summary = spark.sql("""
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS tong_so_don,
        COUNT(r.review_id) AS tong_so_dong_review,
        COUNT(DISTINCT r.review_id) AS so_review_id_khac_nhau,
        COUNT(r.review_id) - COUNT(DISTINCT r.review_id) AS so_luot_review_id_lap
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    LEFT JOIN reviews r ON o.order_id = r.order_id
    GROUP BY c.customer_unique_id
""").cache()

customer_review_summary.createOrReplaceTempView("customer_review_summary")

target_customers = spark.sql("""
    SELECT
        c.customer_unique_id,
        sort_array(collect_set(x.review_id)) AS review_id_thuoc_32_nhom
    FROM review_32 x
    JOIN orders o ON x.order_id = o.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
""").cache()

target_customers.createOrReplaceTempView("target_customers")

print("QUY MÔ THỐNG KÊ TOÀN BỘ KHÁCH HÀNG")
spark.sql("""
    SELECT
        COUNT(*) AS so_khach_hang,
        SUM(tong_so_don) AS tong_so_don,
        SUM(tong_so_dong_review) AS tong_so_dong_review,
        SUM(so_luot_review_id_lap) AS tong_luot_review_id_lap_trong_tung_khach
    FROM customer_review_summary
""").show(truncate=False)

print("THỐNG KÊ KHÁCH HÀNG THUỘC 32 NHÓM")
target_summary = (
    customer_review_summary
    .join(target_customers, "customer_unique_id")
    .orderBy("customer_unique_id")
)

target_summary.show(target_summary.count(), truncate=False)

print("TOÀN BỘ LỊCH SỬ ĐƠN HÀNG CỦA NHỮNG KHÁCH NÀY")
target_history = (
    customer_order_history
    .join(target_customers, "customer_unique_id")
    .orderBy("customer_unique_id", "order_purchase_timestamp", "order_id")
)

target_history.show(target_history.count(), truncate=False)

QUY MÔ THỐNG KÊ TOÀN BỘ KHÁCH HÀNG
+-------------+-----------+-------------------+----------------------------------------+
|so_khach_hang|tong_so_don|tong_so_dong_review|tong_luot_review_id_lap_trong_tung_khach|
+-------------+-----------+-------------------+----------------------------------------+
|96096        |99441      |99224              |814                                     |
+-------------+-----------+-------------------+----------------------------------------+

THỐNG KÊ KHÁCH HÀNG THUỘC 32 NHÓM
+--------------------------------+-----------+-------------------+----------------------+---------------------+----------------------------------+
|customer_unique_id              |tong_so_don|tong_so_dong_review|so_review_id_khac_nhau|so_luot_review_id_lap|review_id_thuoc_32_nhom           |
+--------------------------------+-----------+-------------------+----------------------+---------------------+----------------------------------+
|00172711b30d52eea8b313a7f2cced02|2         

### Cell code 12A: Trực quan hóa lịch sử khách hàng thuộc 32 nhóm

- **Mục đích:** Trực quan hóa số đơn và số lượt review_id lặp trong lịch sử các khách hàng liên quan.
- **Vì sao:** Không nên vẽ trực tiếp từng `customer_unique_id` vì mã dài và khó đọc.
- **Giúp ích:** Quan sát phân bố hành vi mua của nhóm khách có review bất thường.


In [20]:
# Cell code 12A: Trực quan hóa lịch sử khách hàng thuộc 32 nhóm
import hvplot.pandas

target_customer_pd = target_summary.select(
    "customer_unique_id",
    "tong_so_don",
    "tong_so_dong_review",
    "so_review_id_khac_nhau",
    "so_luot_review_id_lap"
).toPandas()

order_count_dist = (
    target_customer_pd.groupby("tong_so_don")
    .size()
    .reset_index(name="Số khách hàng")
    .rename(columns={"tong_so_don": "Tổng số đơn"})
    .sort_values("Tổng số đơn")
)

customer_orders_plot = order_count_dist.hvplot.bar(
    x="Tổng số đơn",
    y="Số khách hàng",
    width=900,
    height=400,
    title="Phân bố số đơn của khách hàng thuộc 32 nhóm",
    ylabel="Số khách hàng"
).opts(hooks=[black_chart_text])

repeat_review_dist = (
    target_customer_pd.groupby("so_luot_review_id_lap")
    .size()
    .reset_index(name="Số khách hàng")
    .rename(columns={"so_luot_review_id_lap": "Số lượt review_id lặp"})
    .sort_values("Số lượt review_id lặp")
)

repeat_review_plot = repeat_review_dist.hvplot.bar(
    x="Số lượt review_id lặp",
    y="Số khách hàng",
    width=900,
    height=400,
    title="Số lượt review_id lặp trong lịch sử từng khách hàng",
    ylabel="Số khách hàng"
).opts(hooks=[black_chart_text])

display(
    pn.Column(
        customer_orders_plot,
        repeat_review_plot,
        styles={"background": "white", "color": "black", "padding": "12px"}
    )
)


Column(styles={'background': 'white', ...})
    [0] HoloViews(Bars, height=400, sizing_mode='fixed', width=900)
    [1] HoloViews(Bars, height=400, sizing_mode='fixed', width=900)

##Nhận xét:
- Đa số review xuất hiện ở đơn đã mua và giao trước đó, rồi cùng review_id lại xuất hiện ở đơn mua sau.

### Cell code 13: So nội dung các dòng review cùng ID

- **Mục đích:** Đối chiếu điểm, tiêu đề, nội dung và thời gian của các dòng cùng review_id.
- **Vì sao:** Nội dung khác nhau và nội dung giống nhau dẫn tới cách xử lý khác nhau.
- **Giúp ích:** Tránh nhầm dòng liên kết lặp thành nhiều bình luận mới.


In [21]:
review_comparison = spark.sql("""
    SELECT
        review_id,
        COUNT(*) AS so_dong,
        COUNT(DISTINCT order_id) AS so_order_id,
        COUNT(DISTINCT to_json(named_struct('v', review_score))) AS so_gia_tri_diem,
        COUNT(DISTINCT to_json(named_struct('v', review_comment_title))) AS so_gia_tri_tieu_de,
        COUNT(DISTINCT to_json(named_struct('v', review_comment_message))) AS so_gia_tri_noi_dung,
        COUNT(DISTINCT to_json(named_struct('v', review_creation_date))) AS so_gia_tri_ngay_tao,
        COUNT(DISTINCT to_json(named_struct('v', review_answer_timestamp))) AS so_gia_tri_luc_tra_loi,
        COUNT(DISTINCT to_json(named_struct(
            'score', review_score,
            'title', review_comment_title,
            'message', review_comment_message,
            'created', review_creation_date,
            'answered', review_answer_timestamp
        ))) AS so_phien_ban_review_khac_nhau
    FROM reviews
    WHERE review_id IS NOT NULL
    GROUP BY review_id
    HAVING COUNT(*) > 1
""").cache()

review_comparison.createOrReplaceTempView("review_comparison")

print("TỔNG HỢP TOÀN BỘ REVIEW_ID LẶP")
spark.sql("""
    SELECT
        COUNT(*) AS so_nhom_review_id_lap,
        SUM(so_dong) AS tong_so_dong,
        SUM(CASE WHEN so_phien_ban_review_khac_nhau = 1 THEN 1 ELSE 0 END) AS nhom_co_review_giong_het,
        SUM(CASE WHEN so_phien_ban_review_khac_nhau > 1 THEN 1 ELSE 0 END) AS nhom_co_review_khac_nhau,
        SUM(CASE WHEN so_gia_tri_diem > 1 THEN 1 ELSE 0 END) AS nhom_khac_diem,
        SUM(CASE WHEN so_gia_tri_tieu_de > 1 THEN 1 ELSE 0 END) AS nhom_khac_tieu_de,
        SUM(CASE WHEN so_gia_tri_noi_dung > 1 THEN 1 ELSE 0 END) AS nhom_khac_noi_dung,
        SUM(CASE WHEN so_gia_tri_ngay_tao > 1 THEN 1 ELSE 0 END) AS nhom_khac_ngay_tao,
        SUM(CASE WHEN so_gia_tri_luc_tra_loi > 1 THEN 1 ELSE 0 END) AS nhom_khac_luc_tra_loi
    FROM review_comparison
""").show(truncate=False)

print("KẾT QUẢ CỦA TỪNG REVIEW_ID LẶP")
review_comparison.orderBy("review_id").show(
    review_comparison.count(), truncate=False
)

print("BẢN GHI GỐC CỦA CÁC NHÓM CÓ TRƯỜNG REVIEW KHÁC NHAU")
different_reviews = spark.sql("""
    SELECT
        r.review_id,
        r.order_id,
        r.review_score,
        r.review_comment_title,
        r.review_comment_message,
        r.review_creation_date,
        r.review_answer_timestamp
    FROM reviews r
    JOIN review_comparison c ON r.review_id = c.review_id
    WHERE c.so_phien_ban_review_khac_nhau > 1
    ORDER BY r.review_id, r.review_creation_date, r.order_id
""")

different_reviews.show(different_reviews.count(), truncate=False)

TỔNG HỢP TOÀN BỘ REVIEW_ID LẶP
+---------------------+------------+------------------------+------------------------+--------------+-----------------+------------------+------------------+---------------------+
|so_nhom_review_id_lap|tong_so_dong|nhom_co_review_giong_het|nhom_co_review_khac_nhau|nhom_khac_diem|nhom_khac_tieu_de|nhom_khac_noi_dung|nhom_khac_ngay_tao|nhom_khac_luc_tra_loi|
+---------------------+------------+------------------------+------------------------+--------------+-----------------+------------------+------------------+---------------------+
|789                  |1603        |789                     |0                       |0             |0                |0                 |0                 |0                    |
+---------------------+------------+------------------------+------------------------+--------------+-----------------+------------------+------------------+---------------------+

KẾT QUẢ CỦA TỪNG REVIEW_ID LẶP
+--------------------------------+---

### Cell code 13A: Trực quan hóa cấu trúc review_id lặp

- **Mục đích:** Trực quan hóa số order_id và số phiên bản nội dung của từng review_id lặp.
- **Vì sao:** Cell 13 có bảng chi tiết 789 nhóm nên rất khó nhìn bằng `.show()`.
- **Giúp ích:** Xác nhận trực quan review_id lặp chủ yếu khác order_id chứ không tạo phiên bản nội dung mới.


In [22]:
# Cell code 13A: Trực quan hóa cấu trúc review_id lặp
import hvplot.pandas

review_compare_pd = review_comparison.select(
    "review_id",
    "so_dong",
    "so_order_id",
    "so_phien_ban_review_khac_nhau"
).toPandas()

order_id_dist = (
    review_compare_pd.groupby("so_order_id")
    .size()
    .reset_index(name="Số review_id")
    .rename(columns={"so_order_id": "Số order_id"})
    .sort_values("Số order_id")
)

order_ids_plot = order_id_dist.hvplot.bar(
    x="Số order_id",
    y="Số review_id",
    width=850,
    height=400,
    title="Số order_id liên kết với mỗi review_id bị lặp",
    ylabel="Số review_id"
).opts(hooks=[black_chart_text])

version_dist = (
    review_compare_pd.groupby("so_phien_ban_review_khac_nhau")
    .size()
    .reset_index(name="Số review_id")
    .rename(columns={"so_phien_ban_review_khac_nhau": "Số phiên bản review khác nhau"})
    .sort_values("Số phiên bản review khác nhau")
)

review_versions_plot = version_dist.hvplot.bar(
    x="Số phiên bản review khác nhau",
    y="Số review_id",
    width=850,
    height=400,
    title="Số phiên bản nội dung trong mỗi review_id bị lặp",
    ylabel="Số review_id"
).opts(hooks=[black_chart_text])

display(
    pn.Column(
        order_ids_plot,
        review_versions_plot,
        styles={"background": "white", "color": "black", "padding": "12px"}
    )
)


Column(styles={'background': 'white', ...})
    [0] HoloViews(Bars, height=400, sizing_mode='fixed', width=850)
    [1] HoloViews(Bars, height=400, sizing_mode='fixed', width=850)

##Nhận xét:
- Có cả 789 review_id lặp đều có điểm số, tiêu đề, nội dung và hai mốc thời gian giống hệt nhau; chỉ order_id khác. Vì vậy, kết quả không chứng minh giả thuyết đây là các review mới hay các phiên bản được chỉnh sửa.